# Nivel 1: Fundamentos de MapReduce

Cada ejercicio se resuelve con dos funciones:

- **`mapper(item)`**: recibe un dato y emite pares `(clave, valor)` con `yield`.
- **`reducer(clave, valores)`**: recibe todos los valores de una misma clave y los combina.

El framework (`mapreduce_framework.py`) se encarga de la fase intermedia (**shuffle**), que agrupa los valores por clave.

```
datos → [MAP] → (clave, valor) → [SHUFFLE] → clave: [valores] → [REDUCE] → resultado
```

In [1]:
from mapreduce_framework import mapreduce, print_results

---
## Ejercicio 1.1: Contador de caracteres

**Objetivo:** contar la frecuencia de cada letra (solo letras, sin distinguir mayúsculas).

- **mapper** → emite `(letra, 1)` por cada letra.
- **reducer** → suma los 1 de cada letra.

In [2]:
text = [
    "MapReduce is a programming model",
    "for processing large data sets",
]


def mapper(line):
    """Emite (letra, 1) por cada letra de la línea."""
    for char in line.lower():
        if char.isalpha():          # ignora espacios y signos
            yield (char, 1)


def reducer(char, counts):
    """Suma las apariciones de una letra."""
    return sum(counts)


results = mapreduce(text, mapper, reducer)
print_results(results, "Frecuencia de letras")

# Caso borde: datos vacíos
print("Datos vacíos:", mapreduce([], mapper, reducer))


Frecuencia de letras
a: 6
r: 6
e: 6
s: 5
m: 4
o: 4
g: 4
p: 3
d: 3
i: 3
c: 2
n: 2
l: 2
t: 2
u: 1
f: 1

Datos vacíos: {}


> **Nota:** el enunciado muestra `'a': 5` y `'e': 4`, pero el valor correcto es **6** para ambas.
> Por ejemplo, la `a` aparece en m**a**preduce, **a**, l**a**rge y d**a**t**a**. El ejemplo del enunciado tiene un error.

---
## Ejercicio 1.2: Palabras largas

**Objetivo:** contar solo las palabras de más de 5 caracteres.

- **mapper** → emite `(palabra, 1)` solo si `len(palabra) > 5`.
- **reducer** → suma.

In [3]:
text = [
    "MapReduce is a programming model",
    "MapReduce processes large volumes of data",
    "The MapReduce model has two main phases",
    "The Map phase transforms the data",
    "The Reduce phase aggregates the results",
]


def mapper(line):
    """Emite (palabra, 1) solo si la palabra tiene más de 5 caracteres."""
    for word in line.lower().split():   # separa la línea en palabras
        word = word.strip('.,!?')       # quita signos de puntuación
        if len(word) > 5:               # solo palabras largas
            yield (word, 1)


def reducer(word, counts):
    """Suma las apariciones de una palabra."""
    return sum(counts)


results = mapreduce(text, mapper, reducer)
print_results(results, "Palabras largas")

print("Datos vacíos:", mapreduce([], mapper, reducer))


Palabras largas
mapreduce: 3
programming: 1
processes: 1
volumes: 1
phases: 1
transforms: 1
reduce: 1
aggregates: 1
results: 1

Datos vacíos: {}


---
## Ejercicio 1.3: Promedio de ventas por producto

**Objetivo:** calcular el monto promedio de venta por producto.

- **mapper** → emite `(producto, monto)`.
- **reducer** → promedio = suma / cantidad.

**Salida esperada:** `Laptop: 1183.3`, `Mouse: 27.5`, `Keyboard: 77.5`

In [4]:
sales = [
    {'product': 'Laptop', 'amount': 1200},
    {'product': 'Mouse', 'amount': 25},
    {'product': 'Laptop', 'amount': 1100},
    {'product': 'Mouse', 'amount': 30},
    {'product': 'Laptop', 'amount': 1250},
    {'product': 'Keyboard', 'amount': 75},
    {'product': 'Keyboard', 'amount': 80},
]


def mapper(sale):
    """Emite (producto, monto)."""
    yield (sale['product'], sale['amount'])


def reducer(product, amounts):
    """Calcula el promedio de ventas de un producto (1 decimal)."""
    if not amounts:                     # evita dividir entre cero
        return 0
    return round(sum(amounts) / len(amounts), 1)


results = mapreduce(sales, mapper, reducer)
print_results(results, "Promedio de ventas por producto")

print("Datos vacíos:", mapreduce([], mapper, reducer))


Promedio de ventas por producto
Laptop: 1183.3
Keyboard: 77.5
Mouse: 27.5

Datos vacíos: {}


---
## Ejercicio 1.4: Estadísticas de temperatura

**Objetivo:** calcular mínimo, máximo y promedio de temperatura por ciudad.

- **mapper** → emite `(ciudad, temperatura)`.
- **reducer** → devuelve un diccionario con `min`, `max` y `avg`.

In [5]:
temperatures = [
    {'city': 'Medellin', 'temperature': 22, 'date': '2026-01-01'},
    {'city': 'Bogota', 'temperature': 14, 'date': '2026-01-01'},
    {'city': 'Medellin', 'temperature': 24, 'date': '2026-01-02'},
    {'city': 'Cali', 'temperature': 28, 'date': '2026-01-01'},
    {'city': 'Bogota', 'temperature': 13, 'date': '2026-01-02'},
    {'city': 'Cali', 'temperature': 30, 'date': '2026-01-02'},
    {'city': 'Medellin', 'temperature': 23, 'date': '2026-01-03'},
    {'city': 'Bogota', 'temperature': 15, 'date': '2026-01-03'},
]


def mapper(record):
    """Emite (ciudad, temperatura)."""
    yield (record['city'], record['temperature'])


def reducer(city, temps):
    """Devuelve un diccionario con 'min', 'max' y 'avg'."""
    return {
        'min': min(temps),
        'max': max(temps),
        'avg': round(sum(temps) / len(temps), 1),
    }


results = mapreduce(temperatures, mapper, reducer)
for city, stats in results.items():
    print(f"{city}: {stats}")

print("Datos vacíos:", mapreduce([], mapper, reducer))

Medellin: {'min': 22, 'max': 24, 'avg': 23.0}
Bogota: {'min': 13, 'max': 15, 'avg': 14.0}
Cali: {'min': 28, 'max': 30, 'avg': 29.0}
Datos vacíos: {}


---
## Ejercicio 1.5: Conteo por categoría

**Objetivo:** contar las ventas por categoría y calcular el total y el promedio.

- **mapper** → emite `(categoría, monto)`.
- **reducer** → devuelve `count`, `total` y `avg`.

In [6]:
sales = [
    {'product': 'Laptop', 'category': 'Electronics', 'amount': 1200},
    {'product': 'Mouse', 'category': 'Electronics', 'amount': 25},
    {'product': 'Desk', 'category': 'Furniture', 'amount': 600},
    {'product': 'Chair', 'category': 'Furniture', 'amount': 350},
    {'product': 'Monitor', 'category': 'Electronics', 'amount': 450},
]


def mapper(sale):
    """Emite (categoría, monto)."""
    yield (sale['category'], sale['amount'])


def reducer(category, amounts):
    """Devuelve un diccionario con 'count', 'total' y 'avg'."""
    total = sum(amounts)
    return {
        'count': len(amounts),
        'total': total,
        'avg': round(total / len(amounts), 1),
    }


results = mapreduce(sales, mapper, reducer)
for category, stats in results.items():
    print(f"{category}: {stats}")

print("Datos vacíos:", mapreduce([], mapper, reducer))

Electronics: {'count': 3, 'total': 1675, 'avg': 558.3}
Furniture: {'count': 2, 'total': 950, 'avg': 475.0}
Datos vacíos: {}
